# Tập cuối F · Robot giao đồ trong nhà: ghép cả 4 mùa 📦

[![Mở trên Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thanhpro82/tu-hoc-robot-tu-hanh/blob/main/tapF/tapF_giaodo.ipynb)

Series **Tự Học Robot Tự Hành** · Tập cuối · ⏱ ~45 phút · Giáo trình: Principles of Robot Autonomy ([máy trạng thái](https://porabook.com/book/finite-state-machines/), [định vị](https://porabook.com/book/robot-localization/), [nhận dạng vật](https://porabook.com/book/object-detection/), [kết hợp cảm biến](https://porabook.com/book/sensor-fusion/))

Robot thức dậy không biết mình ở đâu, lấy đồ ở bếp rồi giao tận tay người nhận. Trong notebook này bạn:

1. Xem số liệu 100 lượt của 5 phiên bản (đủ 4 mùa, và bỏ từng mùa).
2. **Tự viết** 3 mối nối: camera → khoảng cách, ghép camera với LiDAR, máy trạng thái (3 bài), có ô chấm tự động (~15 giây).
3. Vẽ lượt minh hoạ trên bản đồ căn hộ.

Chạy lần lượt từng ô từ trên xuống (Shift + Enter).

## 0. Chuẩn bị (chạy 1 lần)

In [ ]:
import os, sys
if "google.colab" in sys.modules:  # trên Colab: tải code về
    if not os.path.exists("/content/tu-hoc-robot-tu-hanh"):
        !git clone -q https://github.com/thanhpro82/tu-hoc-robot-tu-hanh.git /content/tu-hoc-robot-tu-hanh
    os.chdir("/content/tu-hoc-robot-tu-hanh/tapF")
sys.path[:0] = ["bai_tap", "sim", ".."]

import json
import numpy as np
import giao_do as goc             # robot giao đồ của video
from ve_do_thi import ve          # vẽ hình
from IPython.display import Image, display
from kiem_tra import cham          # bộ chấm
print("Sẵn sàng!")

## 1. Chạy lại số liệu trong video

In [ ]:
tk = json.load(open("sim/ket_qua.txt", encoding="utf8"))["tk"]
for bo, d in tk.items():
    print(f"{bo:9s}: giao thành công {d['ok']}/{d['n']} · hỏng vì {d['ly_do']}")

## 2. Bài tập

Camera lỗ kim: $Z = H f / h$. Ghép cảm biến: camera cho hướng, LiDAR cho khoảng cách. Máy trạng thái: $s_{t+1} = n(s_t, i_t)$.

| Bài | Hàm | Việc |
|---|---|---|
| **1** | `vi_tri_tu_khung(khung)` | khoảng cách, hướng từ khung YOLO |
| **2** | `ghep(uoc, z, Z_cam, huong)` | vị trí người nhận từ camera + LiDAR |
| **3** | `chuyen(s, su_kien, truoc_dung, da_lay)` | hàm chuyển của bộ não |

Thay từng dòng `raise NotImplementedError(...)` bằng code của bạn, chạy ô đó, rồi chạy ô chấm ngay bên dưới (✅ đạt · ❌ sai, kèm gợi ý · ⏳ chưa làm).

In [ ]:
import math

import numpy as np

from giao_do import F_CAM, GOC_TIA, H_NGUOI, N_ANH, R_NGUOI, SO_TIA, goc, vat_la  # noqa: F401  (trong ../sim)


def vi_tri_tu_khung(khung):
    """Bài 1 (camera lỗ kim, tập 2.2): khung = (s, cx, cy, w, h) của YOLO, toạ độ đã chia cho cỡ ảnh N_ANH = 128 px.

    Chiều cao người trong ảnh là h·N_ANH pixel; người thật cao H_NGUOI = 1,7 m; tiêu cự F_CAM = 60 px.
      Z     = H_NGUOI · F_CAM / (h · N_ANH)               (người càng nhỏ trong ảnh càng xa)
      huong = atan((cx − 0.5) · N_ANH / F_CAM)             (> 0: người lệch sang PHẢI ảnh)
    Trả về (Z, huong)."""
    raise NotImplementedError("TODO Bài 1: khoảng cách và hướng từ khung YOLO")


def ghep(uoc, z, Z_cam, huong):
    """Bài 2 (kết hợp cảm biến, tập 3.8): camera biết ĐÓ LÀ NGƯỜI và ở hướng nào, LiDAR đo khoảng cách chuẩn hơn.

    uoc = tư thế ước lượng (x, y, θ); z = 36 tia LiDAR (góc GOC_TIA, từ −π, cách nhau 10°); Z_cam, huong từ Bài 1.
    1. j = tia gần hướng người nhất: argmin |goc(GOC_TIA + huong)|      (người bên phải ảnh = góc âm so với mũi robot)
    2. la = vat_la(uoc, z): tia nào chạm thứ KHÔNG có trên bản đồ.
       Xét 3 tia j−1, j, j+1 (lấy chỉ số mod SO_TIA), giữ các tia có la = True.
    3. Nếu có: chọn tia ngắn nhất i, d = z[i] + R_NGUOI (LiDAR chạm mặt người, tâm ở sau thêm 0,2 m), a = θ + GOC_TIA[i]
       Nếu không: d = Z_cam, a = θ − huong.
    Trả về ((uoc_x + d·cos a, uoc_y + d·sin a), d)."""
    raise NotImplementedError("TODO Bài 2: ghép camera + LiDAR")


def chuyen(s, su_kien, truoc_dung=None, da_lay=False):
    """Bài 3 (máy trạng thái, tập 4.1): hàm chuyển n(s, i) của bộ não robot. Trả về trạng thái tiếp theo.

    su_kien là một trong: "khong", "co_lenh", "da_dinh_vi", "mat_dau", "vat_la", "toi_noi", "duong_trong", "xong_viec",
    "thay_nguoi", "du_gan". Sự kiện không có trong bảng thì GIỮ NGUYÊN trạng thái.
      CHỜ LỆNH   + co_lenh     → ĐỊNH VỊ
      ĐỊNH VỊ    + da_dinh_vi  → ĐI GIAO nếu da_lay, ngược lại ĐI LẤY
      ĐI LẤY, ĐI GIAO, DỪNG + mat_dau → ĐỊNH VỊ               (mất dấu: định vị lại)
      ĐI LẤY, ĐI GIAO + vat_la  → DỪNG
      ĐI LẤY     + toi_noi     → NHẬN ĐỒ
      ĐI GIAO    + toi_noi     → TÌM NGƯỜI
      DỪNG       + duong_trong → truoc_dung                    (quay lại đúng việc đang làm dở)
      NHẬN ĐỒ    + xong_viec   → ĐI GIAO
      TÌM NGƯỜI  + thay_nguoi  → TIẾP CẬN
      TIẾP CẬN   + du_gan      → GIAO
      GIAO       + xong_viec   → XONG"""
    raise NotImplementedError("TODO Bài 3: máy trạng thái")


In [ ]:
cham(globals())

## 3. Xem kết quả

In [ ]:
ve(None, "ket_qua.png")
display(Image("ket_qua.png"))

## 4. Thử thách

Robot đã ở mỗi trạng thái bao lâu trong lượt minh hoạ? (đọc từ `sim/luot_minh_hoa.json`, mỗi mẫu cách nhau 0,2 s)

In [ ]:
from collections import Counter
m = json.load(open("sim/luot_minh_hoa.json", encoding="utf8"))
for s, n in Counter(m["s"]).items():
    print(f"{s:10s} {n * 0.2:5.1f} s")

## Câu hỏi tự kiểm tra

Tự trả lời trước, rồi mở đáp án ở mục *Câu hỏi tự kiểm tra* trong [README của tập](https://github.com/thanhpro82/tu-hoc-robot-tu-hanh/tree/main/tapF).

Code cả series: [github.com/thanhpro82/tu-hoc-robot-tu-hanh](https://github.com/thanhpro82/tu-hoc-robot-tu-hanh)